1. **installation** apache spark et initialisation de la sparksession

In [ ]:
!pip install findspark
!pip install pyspark
import findspark
import pyspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark avec Hadoop sur Google Colab") \
    .getOrCreate()

# verifier si SparkSession est crée correctement
if spark:
    print("SparkSession créée avec succès !")
    print("Spark version:", spark.version)
    print("Master:", spark.sparkContext.master)
    print("Application name:", spark.sparkContext.appName)
else:
    print("Échec de la création de SparkSession !")

SparkSession créée avec succès !
Spark version: 3.5.0
Master: local[*]
Application name: TaxiGraph


*2. ingestion des donnees **

In [ ]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet -O yellow_tripdata_2024-03.parquet


In [ ]:
!wget -q https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -O taxi_zone_lookup.csv

!ls -lh


total 58M
-rw-r--r-- 1 root root    0 Mar 27 02:22 graphframes.jar
drwxr-xr-x 1 root root 4.0K Mar 24 13:34 sample_data
-rw-r--r-- 1 root root  13K Feb 22  2024 taxi_zone_lookup.csv
-rw-r--r-- 1 root root  58M May 22  2024 yellow_tripdata_2024-03.parquet


nous avons telecharge les fichiers contenant les informations pour trajet taxi et zone de taxi et nous avons avec la commande ls -lh lister les fichiers de notre environnement

3. exploration des données

In [ ]:
# lire les données des taxis dans un DataFrame Spark
df_taxi = spark.read.parquet("yellow_tripdata_2024-03.parquet")

# Afficher les 20 premières lignes
df_taxi.show(20)

# Afficher le nombre total d'enregistrements
total_rows_taxi = df_taxi.count()
print(f"Nombre total d'enregistrements dans les données des taxis : {total_rows_taxi}")


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|                 N|         142|         239|           1|        8.6|  3.5|    0.5|       2.


*   df_taxi.show(20) : Affiche les 20 premières lignes du DataFrame df_taxi
*   df_taxi.count() : Retourne le nombre total d'enregistrements dans les données



In [ ]:
# Lire les données des zones desservies dans un DataFrame Spark
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

# Afficher les informations sur les données des zones (schéma)
df_zones.printSchema()

# Afficher la colonne "Borough" sans doublons
df_zones.select("Borough").distinct().show()


root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+-------------+
|      Borough|
+-------------+
|       Queens|
|          EWR|
|      Unknown|
|     Brooklyn|
|Staten Island|
|          N/A|
|    Manhattan|
|        Bronx|
+-------------+





*  df_zones.printSchema() : Affiche le schéma des données
*   df_zones.select("Borough").distinct().show() : Affiche les valeurs uniques dans la colonne "Borough"


transformation


In [ ]:
# Creons une condition pour garder les enregistrements valides
valid_conditions = (df_taxi['fare_amount'] >= 0) & (df_taxi['trip_distance'] >= 0)

# Appliquons le filtre et créons un DataFrame nettoyé
df_taxi_clean = df_taxi.filter(valid_conditions)

# Affichons les premières lignes du DataFrame nettoyé
df_taxi_clean.show(20)

# Comptons le nombre d'enregistrements dans le DataFrame nettoyé
record_count = df_taxi_clean.count()
print(f"Le nombre total d'enregistrements après nettoyage est : {record_count}")


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|                 N|         142|         239|           1|        8.6|  3.5|    0.5|       2.

 la creation de la variable valid_conditions permet de vérifier que fare_amount et trip_distance sont tous les deux supérieurs ou égaux à zéro.

In [ ]:
# Convertir la colonne 'store_and_fwd_flag' en une valeur booléenne
df_taxi_clean = df_taxi_clean.withColumn(
    "store_and_fwd_flag",
    (df_taxi_clean["store_and_fwd_flag"] == "Y")
)

# Afficher les 20 premières lignes pour vérifier la transformation
df_taxi_clean.show(20)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|             false|         142|         239|           1|        8.6|  3.5|    0.5|       2.

df_taxi_clean["store_and_fwd_flag"] == "Y" vérifira si la valeur de la colonne est "Y" et on la transforme en True sinon, elle devient False. ensuite on affiche les 20 premieres lignes.

In [ ]:
from pyspark.sql.functions import when

# Convertir la colonne 'store_and_fwd_flag' en entiers (1 pour 'Y' et 0 pour 'N')
df_taxi_clean = df_taxi_clean.withColumn(
    "store_and_fwd_flag",
    when(df_taxi_clean["store_and_fwd_flag"] == "Y", 1).otherwise(0)
)

# Afficher les 20 premières lignes pour vérifier la transformation
df_taxi_clean.show(20)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|                 0|         142|         239|           1|        8.6|  3.5|    0.5|       2.

when(...) est une  fonction conditionnelle permettant de transformer les valeurs dans la colonne. Si la valeur est "Y", elle devient 1 ; sinon, elle devient 0.
otherwise(0) : Si la condition n’est pas remplie (donc si la valeur est "N"), la colonne sera transformée en 0.

In [ ]:
from pyspark.sql.functions import unix_timestamp

# Premièrement, on convertit les colonnes 'tpep_pickup_datetime' et 'tpep_dropoff_datetime' en timestamps (secondes depuis l'époque Unix)
df_taxi_clean = df_taxi_clean.withColumn(
    "pickup_timestamp", unix_timestamp("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss")
)

df_taxi_clean = df_taxi_clean.withColumn(
    "dropoff_timestamp", unix_timestamp("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss")
)

# Ensuite, on calcule la durée du trajet en soustrayant le timestamp de la prise en charge de celui de la dépose
df_taxi_clean = df_taxi_clean.withColumn(
    "trip_duration", df_taxi_clean["dropoff_timestamp"] - df_taxi_clean["pickup_timestamp"]
)

# Pour vérifier que tout s'est bien passé, on affiche les premières lignes du DataFrame
df_taxi_clean.show(20)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------+-----------------+-------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pickup_timestamp|dropoff_timestamp|trip_duration|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------+-----------------+-------------+
|       1| 2024-03-01 00:18:51|  2024-03

 On commence par transformer les colonnes de dates tpep_pickup_datetime et tpep_dropoff_datetime en timestamps. Ensuite, on fait une simple soustraction entre ces deux timestamps (dépose - prise en charge) pour obtenir la durée du trajet en seconde.enfin on affiche les premières lignes du DataFrame pour vérifier que la nouvelle colonne trip_duration a bien été ajoutée.

5. analyse.


In [ ]:
# Créer une vue SQL temporaire pour le DataFrame des trajets de taxi nettoyé
df_taxi_clean.createOrReplaceTempView("taxi_data")

# Créer une vue SQL temporaire pour le DataFrame des zones desservies
df_zones.createOrReplaceTempView("taxi_zones")

# Vérifier que les vues ont bien été créées (optionnel)
spark.sql("SHOW TABLES").show()


+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|         | taxi_data|       true|
|         |taxi_zones|       true|
+---------+----------+-----------+



createOrReplaceTempView("taxi_data") : Cette méthode crée une vue SQL temporaire nommée taxi_data à partir du DataFrame df_taxi_clean.
createOrReplaceTempView("taxi_zones") :  cette méthode crée une vue temporaire pour le DataFrame df_zones.
spark.sql("SHOW TABLES").show() : Cette commande affiche toutes les tables et vues SQL temporaires existantes dans le session Spark. Elle permet de vérifier que les vues ont bien été créées.

In [ ]:
# Créer la requête SQL pour obtenir les informations
query = """
    SELECT
        td.PULocationID AS pickup_zone_id,
        tz1.Zone AS pickup_zone_name,
        td.DOLocationID AS dropoff_zone_id,
        tz2.Zone AS dropoff_zone_name,
        td.trip_distance,
        td.total_amount
    FROM taxi_data td
    LEFT JOIN taxi_zones tz1 ON td.PULocationID = tz1.LocationID
    LEFT JOIN taxi_zones tz2 ON td.DOLocationID = tz2.LocationID
"""

# Exécuter la requête SQL
result = spark.sql(query)

# Afficher les 20 premières lignes du résultat
result.show(20)


+--------------+--------------------+---------------+--------------------+-------------+------------+
|pickup_zone_id|    pickup_zone_name|dropoff_zone_id|   dropoff_zone_name|trip_distance|total_amount|
+--------------+--------------------+---------------+--------------------+-------------+------------+
|           142| Lincoln Square East|            239|Upper West Side S...|          1.3|        16.3|
|           238|Upper West Side N...|             24|        Bloomingdale|          1.1|        15.2|
|           263|      Yorkville West|             75|   East Harlem South|         0.86|        10.4|
|           164|       Midtown South|            162|        Midtown East|         0.82|       14.19|
|           263|      Yorkville West|              7|             Astoria|          4.9|        30.4|
|           238|Upper West Side N...|            159|       Melrose South|         5.04|        27.9|
|           161|      Midtown Center|            141|     Lenox Hill West|        

- PULocationID et DOLocationID : Ce sont les identifiants des zones de départ et d’arrivée dans les données de trajets.
- taxi_zones : C’est un autre DataFrame qui contient les noms des zones. En associant les identifiants des zones dans les données des trajets avec les noms des zones dans taxi_zones, on peut voir quel nom de zone correspond à chaque ID.
- LEFT JOIN : Ça permet de joindre deux DataFrames. Dans ce cas, on veut joindre taxi_data (les trajets) avec taxi_zones (les zones) pour obtenir les noms des zones de départ et d’arrivée.
- trip_distance et total_amount : Ce sont juste la distance du trajet et le montant total qu’a payé le passager.

In [ ]:
# Créer la requête SQL pour compter les trajets par type de paiement
query_payment_type = """
    SELECT
        payment_type,
        COUNT(*) AS total_trips
    FROM taxi_data
    GROUP BY payment_type
"""

# Exécuter la requête
payment_counts = spark.sql(query_payment_type)

# Afficher le résultat
payment_counts.show()


+------------+-----------+
|payment_type|total_trips|
+------------+-----------+
|           1|    2597067|
|           3|      16390|
|           2|     467301|
|           4|      31518|
|           0|     411888|
+------------+-----------+



-payment_type : C’est la colonne qui contient le type de paiement
-COUNT(*) : Cette fonction compte le nombre d'enregistrements pour chaque type de paiement.
-GROUP BY payment_type : On regroupe les résultats par payment_type, donc chaque ligne correspondra à un type de paiement avec le nombre de trajets associés.

In [ ]:
# Créer un DataFrame réduit avec les 10 000 premiers enregistrements
df_reduit = df_taxi.limit(10000)

# Afficher les premières lignes pour vérifier
df_reduit.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|                 N|         142|         239|           1|        8.6|  3.5|    0.5|       2.

In [ ]:
!pip install graphframes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.6 MB/s eta 0:00:00


In [ ]:
!pip install pyspark

nous avons rencontre des difficultes avec graphframe qui s'averent avoir des problemes de compatibilité

In [ ]:
# Extraire les zones uniques (PULocationID et DOLocationID)
locations = df_reduit.select("PULocationID").union(df_reduit.select("DOLocationID")).distinct()

# Renommer la colonne pour la compatibilité avec GraphFrames
vertices = locations.withColumnRenamed("PULocationID", "id")

# Vérifier les premières zones
vertices.show(5)


+---+
| id|
+---+
|148|
|243|
|137|
| 85|
| 65|
+---+
only showing top 5 rows



In [ ]:
# Créer les arêtes (trajectoires entre les zones avec payment_type)
edges = df_reduit.select(
    col("PULocationID").alias("src"),  # Zone de départ (pickup)
    col("DOLocationID").alias("dst"),  # Zone d'arrivée (dropoff)
    col("payment_type")  # Type de paiement
)

# Vérifier les premières arêtes
edges.show(5)


+---+---+------------+
|src|dst|payment_type|
+---+---+------------+
|142|239|           1|
|238| 24|           1|
|263| 75|           2|
|164|162|           1|
|263|  7|           2|
+---+---+------------+
only showing top 5 rows



In [ ]:
spark.version  # Vérifie la version de Spark


'3.5.0'

page rank a ete fait avec Networks en raison de la difficulte rencontre avec graphframe.Dans ce code, chaque zone (représentée par PULocationID et DOLocationID) est un nœud du graphe.

In [ ]:
import networkx as nx

# Créer un graphe vide
G = nx.DiGraph()

# Ajouter les arêtes au graphe (zone source, zone destination)
edges_data = [(row["PULocationID"], row["DOLocationID"]) for row in df_reduit.collect()]

G.add_edges_from(edges_data)

# Calculer le PageRank
pagerank_scores = nx.pagerank(G, alpha=0.15)

# Afficher les résultats triés
sorted_pagerank = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)
print(sorted_pagerank)


[(231, 0.007088063441631074), (265, 0.007078291645263824), (140, 0.00701740635827151), (137, 0.006222546294697411), (42, 0.006139359656782322), (75, 0.006099954073025569), (130, 0.005845349192930031), (107, 0.005708484529960995), (229, 0.005669549559663132), (227, 0.0055632872563431135), (132, 0.005558433991402701), (237, 0.0054761559430991195), (129, 0.005455792019475142), (205, 0.0054225538114835055), (230, 0.005422423741551524), (49, 0.005422406070265837), (197, 0.0054101624639497576), (48, 0.005381106862369666), (248, 0.005375614173664778), (45, 0.005329194629591856), (228, 0.0052930269266429795), (145, 0.005287043725940442), (79, 0.00517378508783869), (260, 0.005156705524949504), (112, 0.005146586694355555), (238, 0.005134955273058434), (161, 0.005114850638679917), (263, 0.00510815818609073), (19, 0.005103256606521559), (97, 0.005020548469160004), (43, 0.005017508246631841), (162, 0.004992696532211722), (88, 0.0049523333835523), (170, 0.0049373342073580784), (226, 0.00492539302347